## **Feature Engineering — Churn em E-commerce**

Construção das features para o modelo preditivo a partir dos dados brutos de clientes e transações.

#### **Features criadas:**
- **RFV** — Recência, Frequência e Valor
- **Comportamentais** — compras por categoria, método de pagamento preferido
- **Proporções** — participação de cada categoria no mix de compras
- **Engenharia** — ticket médio relativo, flag de cliente sem compra

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_RAW  = Path('../data/raw')
DATA_PROC = Path('../data/processed')
DATA_PROC.mkdir(exist_ok=True)

## **Carregar Dados:**

In [2]:
clientes   = pd.read_csv(DATA_RAW / 'clientes.csv')
transacoes = pd.read_csv(DATA_RAW / 'transacoes.csv')
transacoes['data_compra'] = pd.to_datetime(transacoes['data_compra'])

print(f'Clientes  : {clientes.shape}')
print(f'Transações: {transacoes.shape}')
print(f'Taxa de churn: {clientes["churn"].mean():.2%}')

Clientes  : (10000, 6)
Transações: (72294, 5)
Taxa de churn: 22.26%


## **Features RFV (Recência, Frequência, Valor):**

In [3]:
# Data de referência: dia seguinte à última transação
data_ref = transacoes['data_compra'].max() + pd.Timedelta(days=1)

rfv = transacoes.groupby('id_cliente').agg(
    recencia_dias      =('data_compra', lambda x: (data_ref - x.max()).days),
    frequencia_compras =('id_cliente', 'count'),
    valor_total        =('valor', 'sum'),
    ticket_medio       =('valor', 'mean'),
    std_valor          =('valor', 'std'),
).round(2).reset_index()

# Merge e tratamento de clientes sem compras
df = clientes.merge(rfv, on='id_cliente', how='left')
df['recencia_dias']      = df['recencia_dias'].fillna(999)   # nunca comprou > máxima recência
df['frequencia_compras'] = df['frequencia_compras'].fillna(0)
df['valor_total']        = df['valor_total'].fillna(0)
df['ticket_medio']       = df['ticket_medio'].fillna(0)
df['std_valor']          = df['std_valor'].fillna(0)
df['cliente_sem_compra'] = (df['frequencia_compras'] == 0).astype(int)

n_sem_compra = df['cliente_sem_compra'].sum()
print(f'RFV criado | Clientes sem compra: {n_sem_compra} ({n_sem_compra/len(df):.1%})')
print(f'Taxa de churn (sem compra): {df[df["cliente_sem_compra"]==1]["churn"].mean():.1%}')
print(f'Taxa de churn (com compra): {df[df["cliente_sem_compra"]==0]["churn"].mean():.1%}')

RFV criado | Clientes sem compra: 0 (0.0%)
Taxa de churn (sem compra): nan%
Taxa de churn (com compra): 22.3%


## **Features Comportamentais:**

In [4]:
# Compras por categoria
cat_pivot = transacoes.groupby(['id_cliente','categoria']).size()\
    .unstack(fill_value=0).reset_index()
cat_pivot.columns = ['id_cliente'] + [
    f'compras_{c.lower()}' for c in cat_pivot.columns[1:]
]

# Método de pagamento preferido
pag_pref = transacoes.groupby(['id_cliente','metodo_pagamento']).size()\
    .unstack(fill_value=0)
pag_pref = pag_pref.idxmax(axis=1).reset_index()
pag_pref.columns = ['id_cliente','metodo_preferido']

# Dias entre primeira e última compra
tempo = transacoes.groupby('id_cliente')['data_compra'].agg(['min','max']).reset_index()
tempo['dias_entre_compras'] = (tempo['max'] - tempo['min']).dt.days
tempo = tempo[['id_cliente','dias_entre_compras']]

# Merge
df = df.merge(cat_pivot, on='id_cliente', how='left')
df = df.merge(pag_pref,  on='id_cliente', how='left')
df = df.merge(tempo,     on='id_cliente', how='left')

# Preencher nulos de categorias
CATEGORIAS = [c for c in df.columns if c.startswith('compras_')]
for cat in CATEGORIAS:
    df[cat] = df[cat].fillna(0)

df['metodo_preferido']   = df['metodo_preferido'].fillna('Sem compras')
df['dias_entre_compras'] = df['dias_entre_compras'].fillna(0)

print(f'Categorias criadas: {CATEGORIAS}')

Categorias criadas: ['compras_beleza', 'compras_casa', 'compras_eletrônicos', 'compras_esportes', 'compras_moda']


## **Features de Proporção e Engenharia:**

In [5]:
# Proporção de compras por categoria
for cat in CATEGORIAS:
    prop_col = f'prop_{cat}'
    df[prop_col] = df[cat] / df['frequencia_compras'].replace(0, np.nan)
    df[prop_col] = df[prop_col].fillna(0)

# Ticket médio relativo à média geral
media_ticket_geral     = df[df['ticket_medio'] > 0]['ticket_medio'].mean()
df['ticket_medio_rel'] = df['ticket_medio'] / media_ticket_geral

print(f'Total de features criadas: {df.shape[1]}')
print(f'Features de proporção: {[c for c in df.columns if c.startswith("prop_")]}')

Total de features criadas: 25
Features de proporção: ['prop_compras_beleza', 'prop_compras_casa', 'prop_compras_eletrônicos', 'prop_compras_esportes', 'prop_compras_moda']


## **Preparação para Modelagem:**

In [6]:
# Features numéricas
FEATURES_NUM = [
    'idade', 'recencia_dias', 'frequencia_compras', 'valor_total',
    'ticket_medio', 'std_valor', 'dias_entre_compras', 'ticket_medio_rel',
    'cliente_sem_compra'
] + CATEGORIAS + [f'prop_{c}' for c in CATEGORIAS]

# Features categóricas
FEATURES_CAT = ['genero', 'cidade', 'canal_aquisicao', 'metodo_preferido']

# One-hot encoding (drop_first para evitar multicolinearidade)
df_encoded = pd.get_dummies(df[FEATURES_CAT], drop_first=True)

# Montar X e y — sem escalonamento 
# O escalonamento será feito dentro do pipeline de modelagem para evitar data leakage
X = pd.concat([df[FEATURES_NUM], df_encoded], axis=1)
y = df['churn']

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Taxa de churn: {y.mean():.2%}')
print(f'\nFeatures numéricas: {len(FEATURES_NUM)}')
print(f'Features após encoding: {X.shape[1]}')

# Correlações com churn
corrs = pd.Series(
    [abs(X[col].corr(y)) for col in X.columns],
    index=X.columns
).sort_values(ascending=False)
print('\nTop 10 Correlações com Churn')
print(corrs.head(10).round(4).to_string())

X shape: (10000, 29)
y shape: (10000,)
Taxa de churn: 22.26%

Features numéricas: 19
Features após encoding: 29

Top 10 Correlações com Churn
frequencia_compras           0.2571
valor_total                  0.2191
compras_moda                 0.1673
canal_aquisicao_Indicação    0.1595
cidade_SP                    0.1495
dias_entre_compras           0.1355
compras_casa                 0.1235
compras_esportes             0.1216
compras_eletrônicos          0.1204
compras_beleza               0.1183


## **Salvar Dados Processados:**

In [7]:
# Salvar sem escalonamento — o scaler vai dentro do pipeline
X.to_csv(DATA_PROC / 'X_features.csv', index=False)
y.to_csv(DATA_PROC / 'y_target.csv',   index=False)
df.to_csv(DATA_PROC / 'clientes_com_features.csv', index=False)

print('Arquivos salvos:')
print(f'  data/processed/X_features.csv          — {X.shape}')
print(f'  data/processed/y_target.csv             — {y.shape}')
print(f'  data/processed/clientes_com_features.csv — {df.shape}')

Arquivos salvos:
  data/processed/X_features.csv          — (10000, 29)
  data/processed/y_target.csv             — (10000,)
  data/processed/clientes_com_features.csv — (10000, 25)
